# 03 Sample Size (Synopsis Lock)

Reproduces Green / power calculations from the submitted synopsis.

**Final N = max(RQ1–RQ4) = 347**


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from project_paths import load_env, ensure_output_dirs
ROOT = load_env()
PATHS = ensure_output_dirs(ROOT)
print("ROOT", ROOT)
print({k: str(v) for k, v in PATHS.items()})


In [ ]:
import math
import pandas as pd
import numpy as np

def green(k, alpha=0.05):
    # Green's rule N >= 50 + 8k
    return 50 + 8 * k

def corr_power_n(r=0.15, alpha=0.05, power=0.80):
    # Fisher z sample size used in synopsis for RQ3
    z_a = 1.96
    z_b = 0.84
    fisher = 0.5 * math.log((1 + r) / (1 - r))
    n = ((z_a + z_b) / fisher) ** 2 + 3
    return math.ceil(n)

rows = [
    {"Research Question": "RQ1", "Method": "Green", "Key parameters": "alpha=.05; k=6", "Minimum N": green(6)},
    {"Research Question": "RQ2", "Method": "Green", "Key parameters": "alpha=.05; k=8", "Minimum N": green(8)},
    {"Research Question": "RQ3", "Method": "Correlation power (Fisher z)", "Key parameters": "alpha=.05; power=.80; r=.15", "Minimum N": corr_power_n()},
    {"Research Question": "RQ4", "Method": "Green", "Key parameters": "alpha=.05; k=10", "Minimum N": green(10)},
]
tbl = pd.DataFrame(rows)
final_n = int(tbl["Minimum N"].max())
tbl.loc[len(tbl)] = {"Research Question": "Final N", "Method": "max(RQ1-RQ4)", "Key parameters": "Protects weakest-powered RQ", "Minimum N": final_n}
assert final_n == 347, final_n
tbl.to_csv(PATHS["tables"] / "sample_size.csv", index=False)
tbl


In [ ]:
df = pd.read_csv(PATHS["processed"] / "daily_panel_clean.csv", parse_dates=["date"])
available_n = len(df)
print("Available N:", available_n)
print("Final N required:", 347)
print("Meets Final N:", available_n >= 347)
pd.DataFrame([{
    "available_n": available_n,
    "final_n_required": 347,
    "meets_requirement": available_n >= 347,
}]).to_csv(PATHS["tables"] / "sample_size_vs_available.csv", index=False)
